# Lesson 13: Building Low-Code AI Applications

In this notebook, you will build two low-code AI apps on Google Colab Free Tier using **`Gradio` + `sqlite3` + `google-genai`**:
1. **Student Assignment Tracker & AI Feedback Assistant**
2. **Automated Invoice Extraction Pipeline** (`ID`, `Amount`, `Date`, `Vendor`, `Status`, `Supplier_Email`)

In [ ]:
!pip install -q -U gradio google-genai pydantic pandas

In [ ]:
import os, sqlite3, json
from pydantic import BaseModel
from google import genai
from google.genai import types

class InvoiceRecord(BaseModel):
    id: str
    amount: float
    date: str
    vendor_name: str
    status: str
    supplier_email: str

raw_invoice_text = "INVOICE #INV-881 | Date: 2026-11-20 | Vendor: BrightLab Supplies (ap@brightlab.io) | Total Due: $1,250.00 | Status: Pending Approval"

key = None
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
key = key or os.environ.get("GEMINI_API_KEY")

if key:
    client = genai.Client(api_key=key)
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Extract invoice fields from: {raw_invoice_text}",
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=InvoiceRecord),
    )
    print("Extracted JSON:", resp.text)
else:
    print('[Offline Demo Mode] {"id": "INV-881", "amount": 1250.0, "date": "2026-11-20", "vendor_name": "BrightLab Supplies", "status": "Pending Approval", "supplier_email": "ap@brightlab.io"}')

In [ ]:
import gradio as gr
import pandas as pd

assignments_df = pd.DataFrame([
    {"ID": "ASG-1", "Student": "Maya Lin", "Topic": "Prompt Engineering", "Status": "Graded", "Score": 95},
    {"ID": "ASG-2", "Student": "Liam Patel", "Topic": "RAG Pipelines", "Status": "Pending", "Score": None},
])

def filter_assignments(status_filter):
    if status_filter == "All":
        return assignments_df
    return assignments_df[assignments_df["Status"] == status_filter]

with gr.Blocks(title="Student Assignment Tracker") as demo:
    gr.Markdown("### Low-Code Student Assignment Tracker")
    status = gr.Dropdown(["All", "Graded", "Pending"], value="All", label="Filter by Status")
    table = gr.Dataframe(value=assignments_df)
    status.change(filter_assignments, inputs=status, outputs=table)

print("Gradio app defined! Run demo.launch(share=True) in Colab to open the interactive UI.")